In [ ]:
import json
with open("data/category.json", "r") as f:
    category_weights = json.load(f)

    total_tasks = 0
    for category, weights in category_weights.items():
        total_tasks += len(weights) * 8

    print(f"Total aggregated experiments to run: {total_tasks}")
    current_task = 0

Total aggregated experiments to run: 64


In [ ]:
import pandas as pd
import numpy as np
from typing import Tuple
from src.data_loader import load_csvs
from src.sampler import sample_by_step
from src.utils import get_logger

logger = get_logger(__name__)


def build_dataset_from_source(
    source: str,
    time_col: str = "__time",
    speed_col: str = "speed",
    power_col: str = "power",
    acc_col: str = "accumulated_usage",
    min_rows_in_window: int = 10,
    max_interval_minutes: int = 15,
    min_coverage_rate: float = 0.5,
) -> tuple[pd.DataFrame, dict]:

    df = load_csvs(source)

    sampled, metadata = sample_by_step(
        df,
        time_col=time_col,
        speed_col=speed_col,
        power_col=power_col,
        acc_col=acc_col,
        min_rows_in_window=min_rows_in_window,
        max_interval_minutes=max_interval_minutes,
        min_coverage_rate=min_coverage_rate,
    )
    # drop samples with NaN usage_rate
    sampled = sampled.dropna(
        subset=["usage_rate", "speed_mean", "power_mean"]
    ).reset_index(drop=True)
    return sampled, metadata

sampled, metadata=build_dataset_from_source('data/LZGJL484XPX038050.csv')

Loading CSV(s) from path: data/LZGJL484XPX038050.csv
number of groups before trim: 5
group sizes before trim: [1, 1, 1, 1, 1]
number of groups after trim: 0
group sizes after trim: []


KeyError: ['usage_rate', 'speed_mean', 'power_mean']

In [2]:
import os
import json
target_dir_path = 'outputs/results'
best_models = {}
if os.path.isdir(target_dir_path) :

  for file in os.listdir(target_dir_path) :

    if file.endswith('.json'):
      with open(os.path.join(target_dir_path, file), 'r') as f:
        data = json.load(f)
        min_mape = float('inf')
        best_model = None
        for result in data['results']:
          this_model = result['model']
          this_mape = result['full_model_metrics']['mape']
          if this_mape < min_mape:
            min_mape = this_mape
            best_model = this_model

        if best_model in best_models:
          best_models[best_model] += 1
        else:
          best_models[best_model] = 1
        print(f'Best model for {file:30}: {best_model:10} with MAPE: {min_mape:.3f}%')
  print('Overall best models frequency:')
  for model, freq in best_models.items():
    print(f'{model:10}: {freq}')

Best model for LZGCR2868PX033622.json        : polynomial with MAPE: 8.054%
Best model for LZGCR2X60PX029467.json        : gam        with MAPE: 6.128%
Best model for LZGCR2R6XNX068660.json        : gam        with MAPE: 14.553%
Best model for LZGCD2L1XPX018811.json        : linear     with MAPE: 6.307%
Best model for LZGCR2U62PX015106.json        : gam        with MAPE: 9.037%
Best model for LZGJL484XPX042129.json        : polynomial with MAPE: 10.016%
Best model for LZGCR286XPX014960.json        : polynomial with MAPE: 8.758%
Best model for LZGCR2Y61PX066221.json        : rf         with MAPE: 9.491%
Best model for LZGCD2L12PX023680.json        : linear     with MAPE: 4.917%
Best model for LZGJLG846PX051993.json        : gam        with MAPE: 6.421%
Best model for LZGJL3Z48PX029064.json        : ridge      with MAPE: 6.263%
Best model for diesel_12000_15000kg.json     : gam        with MAPE: 12.164%
Best model for LZGCR2X61PX026349.json        : polynomial with MAPE: 6.680%
Best mode

In [9]:
from src.pipeline import features_labels_from_sampled,build_dataset_from_source


sampled, metadata = build_dataset_from_source('data/LZGCR2U61PX055290.csv')
if sampled.empty:
    logger.error("No valid samples generated. Check column names and data.")


X, y = features_labels_from_sampled(sampled)
# print X and y
print("Features (X):")
print(X)
print("\nLabels (y):")
print(y)

Loading CSV(s) from path: data/LZGCR2U61PX055290.csv
number of groups before trim: 310
group sizes before trim: [5, 89, 26, 39, 3, 5, 111, 2, 3, 21, 37, 25, 20, 13, 23, 4, 36, 8, 27, 13, 18, 47, 102, 57, 6, 124, 23, 99, 10, 87, 23, 2, 1, 13, 173, 12, 42, 172, 76, 9, 57, 4, 56, 36, 15, 15, 12, 12, 13, 15, 6, 13, 41, 13, 16, 45, 113, 77, 12, 5, 56, 45, 18, 18, 34, 14, 64, 173, 14, 89, 100, 93, 168, 17, 26, 31, 6, 18, 49, 88, 84, 78, 39, 82, 82, 222, 21, 14, 134, 127, 22, 122, 88, 50, 55, 88, 59, 38, 40, 68, 69, 14, 1, 21, 1, 57, 2, 63, 77, 4, 37, 30, 45, 1, 47, 7, 2, 37, 8, 13, 7, 39, 21, 76, 160, 9, 1, 37, 7, 314, 75, 93, 64, 104, 45, 24, 42, 103, 121, 51, 30, 7, 9, 34, 35, 68, 20, 114, 10, 44, 57, 8, 5, 14, 57, 45, 115, 96, 17, 129, 7, 20, 19, 55, 74, 62, 14, 1, 33, 24, 2, 20, 41, 53, 38, 33, 49, 15, 9, 12, 29, 24, 49, 12, 22, 57, 66, 27, 33, 63, 3, 29, 18, 39, 36, 12, 5, 9, 39, 105, 23, 51, 13, 54, 45, 63, 4, 1, 47, 113, 101, 86, 22, 33, 110, 92, 58, 83, 5, 12, 47, 12, 98, 36, 14, 49,